# Заполнение `baseline_predictions.csv`

Задаем тривиальный baseline и считаем его вердикт для каждого объекта. 
Выбрано тривиальногое правило сравнения: <b>«длиннее — лучше»</b>.

- Если `output_1` длиннее, тогда baseline вердикт `A`;
- Если `output_2` длиннее, тогда baseline вердикт `B`;
- Если длина равна — `TIE`.

In [1]:
import pandas as pd
import json
from pathlib import Path

AUDIT_DIR = Path("audit")
SLICE_PATH = AUDIT_DIR / "slice.jsonl"
BASELINE_PATH = AUDIT_DIR / "baseline_predictions.csv"

with open(SLICE_PATH, "r", encoding="utf-8") as f:
    slice_records = [json.loads(line) for line in f if line.strip()]

slice_df = pd.DataFrame(slice_records)

# Считаем длину ответов в символах
slice_df["len_output_1"] = slice_df["output_1"].astype(str).str.len()
slice_df["len_output_2"] = slice_df["output_2"].astype(str).str.len()

# Простое правило: более длинный ответ лучше
def length_baseline(row):
    if row["len_output_1"] > row["len_output_2"]:
        return "A"
    elif row["len_output_2"] > row["len_output_1"]:
        return "B"
    else:
        return "TIE"

slice_df["verdict"] = slice_df.apply(length_baseline, axis=1)

# Сохраняем baseline_predictions.csv
baseline_df = slice_df[["id", "verdict"]]
baseline_df.to_csv(BASELINE_PATH, index=False, encoding="utf-8-sig")

print(f"Создан файл: {BASELINE_PATH.resolve()}")
print(f"Объектов: {len(baseline_df)}")
print(baseline_df["verdict"].value_counts())

Создан файл: C:\My Documents\Магистратура\Семестр 3\Нейроные сети (RL)\Практики\Практика 3\audit\baseline_predictions.csv
Объектов: 52
verdict
B      37
A      10
TIE     5
Name: count, dtype: int64
